# Capítulo 8 — Engenharia de conjuntos de dados

**Livro:** começa na página **657** do PDF.

**Gabarito**: soluções dos exercícios do caderno de prática. Tente sozinho antes de olhar. Cada chamada ao modelo custa centavos de centavo.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # onde ficam ia.py e os exercícios
from ia import *               # cliente, perguntar(), conversar(), embeddings(), gasto() ...
print("Pronto. Modelo padrão:", MODELO)

### Síntese de dados com inteligência artificial · PDF p. 699

#### Exercício: Gerar dados sintéticos com o modelo

Peça ao modelo 8 perguntas de clientes de uma farmácia, **variadas** (tema, tom, tamanho), em JSON (`{'perguntas': [...]}`). Depois verifique: quantas são únicas? Alguma é parecida demais com outra?

In [ ]:
import json
saida = perguntar("Gere 8 perguntas variadas (tema, tom e tamanho) que clientes fariam a uma farmácia. "
                  "Responda em JSON: {\"perguntas\": [...]}", temperature=1, response_format={"type": "json_object"})
geradas = json.loads(saida)["perguntas"]
print(len(geradas), "geradas,", len(set(geradas)), "únicas")
for g in geradas:
    print("-", g)

### Inspecionar dados · PDF p. 717

#### Exercício: Olhar um conjunto de instruções

Use a lista `DADOS` abaixo (pares instrução/resposta). Com pandas, veja o tamanho médio das respostas, quais estão vazias ou curtas demais e quais repetem.

```python
DADOS = [
    {'instrucao': 'O que é MRP?', 'resposta': 'Planejamento de necessidades de materiais.'},
    {'instrucao': 'O que é MRP?', 'resposta': 'Planejamento de necessidades de materiais.'},
    {'instrucao': 'Defina lead time', 'resposta': ''},
    {'instrucao': 'O que é estoque de segurança?', 'resposta': 'Reserva para cobrir variações de demanda e de entrega.'},
    {'instrucao': 'Explique ruptura', 'resposta': 'ok'},
]
```

In [ ]:
import pandas as pd
DADOS = [
    {"instrucao": "O que é MRP?", "resposta": "Planejamento de necessidades de materiais."},
    {"instrucao": "O que é MRP?", "resposta": "Planejamento de necessidades de materiais."},
    {"instrucao": "Defina lead time", "resposta": ""},
    {"instrucao": "O que é estoque de segurança?", "resposta": "Reserva para cobrir variações de demanda e de entrega."},
    {"instrucao": "Explique ruptura", "resposta": "ok"},
]
df = pd.DataFrame(DADOS)
df["palavras"] = df.resposta.str.split().str.len()
print("Média de palavras:", df.palavras.mean())
print("Duplicadas:", int(df.duplicated().sum()))
df[df.palavras < 3]   # vazias ou curtas demais

### Remover dados duplicados · PDF p. 720

#### Exercício: Duplicatas exatas e quase iguais

Remova duplicatas exatas de uma lista de perguntas e depois ache **quase duplicatas** com similaridade de embeddings acima de 0,9.

In [ ]:
import numpy as np
perguntas = ["Qual o prazo de entrega?", "Qual o prazo de entrega?", "Em quanto tempo chega meu pedido?",
             "Vocês aceitam Pix?", "Posso pagar com Pix?", "Qual o horário da loja?"]
unicas = list(dict.fromkeys(perguntas))
print("Sem duplicatas exatas:", len(perguntas), "->", len(unicas))
m = similaridade(embeddings(unicas), embeddings(unicas))
for i in range(len(unicas)):
    for j in range(i + 1, len(unicas)):
        if m[i, j] > 0.9:
            print(f"{m[i, j]:.2f}  {unicas[i]!r} ~ {unicas[j]!r}")

### Dados de formato · PDF p. 723

#### Exercício: Formatar para ajuste fino (JSONL de chat)

Transforme os pares instrução/resposta válidos do exercício de inspeção no formato de chat usado em ajuste fino (`{'messages': [{'role': 'user', ...}, {'role': 'assistant', ...}]}`) e salve em `treino.jsonl`, uma linha por exemplo.

In [ ]:
import json
pares = [("O que é MRP?", "Planejamento de necessidades de materiais."),
         ("O que é estoque de segurança?", "Reserva para cobrir variações de demanda e de entrega.")]
with open("treino.jsonl", "w", encoding="utf-8") as f:
    for pergunta, resposta in pares:
        linha = {"messages": [{"role": "user", "content": pergunta}, {"role": "assistant", "content": resposta}]}
        f.write(json.dumps(linha, ensure_ascii=False) + "\n")
print(open("treino.jsonl", encoding="utf-8").read())

## Quanto gastei

In [ ]:
print(gasto())  # quanto esta sessão custou